In [1]:
# ==========================================================
# BIMODAL BRANCH PREDICTOR WITH FEDERATED LEARNING
# Based on: "Two-Level Adaptive Training Branch Prediction" by Yeh & Patt (1991)
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple, Optional

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# BIMODAL CONFIGURATION
# ==========================================================
class BimodalConfig:
    def __init__(self, table_size: int = 2048, counter_bits: int = 2):
        """
        Bimodal branch predictor configuration
        
        Args:
            table_size: Number of entries in prediction table (must be power of 2)
            counter_bits: Number of bits per counter (typically 2)
        """
        self.table_size = table_size
        self.counter_bits = counter_bits
        self.counter_max = (1 << counter_bits) - 1
        self.midpoint = 1 << (counter_bits - 1)  # Midpoint for prediction
        
        # Table size must be power of 2 for efficient indexing
        assert (table_size & (table_size - 1)) == 0, "Table size must be power of 2"
        
    def get_total_storage_bits(self) -> int:
        """Calculate total storage in bits"""
        return self.table_size * self.counter_bits
    
    def get_total_storage_bytes(self) -> int:
        """Calculate total storage in bytes"""
        return self.get_total_storage_bits() // 8
    
    def get_total_storage_kb(self) -> float:
        """Calculate total storage in KB"""
        return self.get_total_storage_bits() / (8 * 1024)

# ==========================================================
# DATA LOADING
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.int64)  # Branch PC
    y = df.iloc[:, -1].values.astype(np.int64)   # Branch outcome (0/1)
    return X, y

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

# ==========================================================
# BIMODAL PREDICTOR
# ==========================================================
class BimodalPredictor:
    """
    Bimodal Branch Predictor (2-bit Saturating Counter Predictor)
    
    The simplest branch predictor that uses the branch PC to index into
    a table of 2-bit saturating counters. Each counter tracks the bias
    of a specific static branch.
    
    Counter values:
        For 2-bit: 0 = Strongly Not Taken
                   1 = Weakly Not Taken  
                   2 = Weakly Taken
                   3 = Strongly Taken
    
    Prediction: Taken if counter >= midpoint (2 for 2-bit)
    """
    def __init__(self, config: BimodalConfig):
        self.config = config
        
        # Prediction table: saturating counters indexed by PC
        # Initialize to weakly taken (midpoint) for optimistic prediction
        self.prediction_table = [config.midpoint] * config.table_size
        
        # Statistics
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0
        
    def _get_index(self, pc: int) -> int:
        """
        Compute index into prediction table using PC
        Simple modulo operation (bitmask if table size is power of 2)
        """
        if self.config.table_size & (self.config.table_size - 1) == 0:
            # Power of 2: use bitmask for efficiency
            return pc & (self.config.table_size - 1)
        else:
            return pc % self.config.table_size
    
    def predict(self, pc: int) -> bool:
        """
        Predict branch direction based on PC-indexed counter
        Returns: True = taken, False = not taken
        """
        self.predictions += 1
        
        idx = self._get_index(pc)
        counter = self.prediction_table[idx]
        
        # Prediction: taken if counter >= midpoint
        return counter >= self.config.midpoint
    
    def update(self, pc: int, taken: bool):
        """
        Update predictor with actual branch outcome
        Uses saturating counter update
        """
        idx = self._get_index(pc)
        counter = self.prediction_table[idx]
        
        # Saturating counter update
        if taken:
            if counter < self.config.counter_max:
                counter += 1
        else:
            if counter > 0:
                counter -= 1
        
        self.prediction_table[idx] = counter
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """
        Make prediction and update with actual outcome
        Returns: prediction made before update
        """
        pred = self.predict(pc)
        
        # Update statistics
        if pred == actual:
            self.correct_predictions += 1
        else:
            self.mispredictions += 1
        
        # Update predictor state
        self.update(pc, actual)
        
        return pred
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy percentage"""
        if self.predictions == 0:
            return 0.0
        return (self.correct_predictions / self.predictions) * 100
    
    def get_mpki(self, instructions: int) -> float:
        """Calculate MPKI (Mispredictions Per Kilo Instructions)"""
        if instructions == 0:
            return 0.0
        return (self.mispredictions / instructions) * 1000
    
    def reset(self):
        """Reset predictor state to initial values"""
        self.prediction_table = [self.config.midpoint] * self.config.table_size
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0
    
    def get_table_distribution(self) -> dict:
        """Get distribution of counter values in the prediction table"""
        counts = {}
        for i in range(self.config.counter_max + 1):
            counts[i] = self.prediction_table.count(i)
        return counts

# ==========================================================
# BIMODAL MODEL WRAPPER FOR FEDERATED LEARNING
# ==========================================================
class BimodalModel(nn.Module):
    """
    Wrapper to make Bimodal compatible with PyTorch training loop
    Note: This is not a neural network; we're using PyTorch's structure
    for federated learning orchestration only
    """
    def __init__(self, config: BimodalConfig):
        super().__init__()
        self.config = config
        self.predictor = BimodalPredictor(config)
        
        # For compatibility with PyTorch optimizer
        self.parameters_list = nn.ParameterList([
            nn.Parameter(torch.tensor(0.0))  # Dummy parameter
        ])
        
    def forward(self, x):
        """
        Forward pass for batch prediction
        x: batch of branch PCs
        """
        batch_size = x.size(0)
        predictions = []
        
        for i in range(batch_size):
            pc = x[i, 0].item()  # Assuming PC is first column
            pred = self.predictor.predict(pc)
            predictions.append(1.0 if pred else 0.0)
        
        return torch.tensor(predictions, device=x.device)
    
    def update(self, pc: int, taken: bool):
        """Update predictor state"""
        self.predictor.update(pc, taken)
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """Predict and update with actual outcome"""
        return self.predictor.predict_and_update(pc, actual)
    
    def reset_state(self):
        """Reset predictor state for new program phase"""
        self.predictor.reset()
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy"""
        return self.predictor.get_accuracy()
    
    def get_mpki(self, instructions: int) -> float:
        """Get MPKI metric"""
        return self.predictor.get_mpki(instructions)

# ==========================================================
# FEDERATED LEARNING WITH BIMODAL
# ==========================================================
class FederatedBimodal:
    """Federated learning with Bimodal predictors"""
    
    def __init__(self, config: BimodalConfig):
        self.config = config
        self.global_model = BimodalModel(config)
        
    def train_client(self, model: BimodalModel, data_loader: DataLoader):
        """
        Train client on their data stream
        Bimodal training is just updating counters with actual outcomes
        """
        model.reset_state()  # Reset for new program phase
        
        for Xb, yb in data_loader:
            for i in range(Xb.size(0)):
                pc = Xb[i, 0].item()
                actual = yb[i].item()
                
                # Predict and update with actual outcome
                model.predict_and_update(pc, actual)
        
        return model
    
    def aggregate_models(self, client_models: List[BimodalModel]):
        """
        Aggregate client models by averaging prediction table counters
        This is federated averaging for bimodal predictors
        """
        # Create new aggregated model
        agg_model = BimodalModel(self.config)
        num_clients = len(client_models)
        
        # Average each prediction table entry
        for i in range(self.config.table_size):
            sum_counter = 0
            for client_model in client_models:
                sum_counter += client_model.predictor.prediction_table[i]
            
            # Average and round to nearest integer
            avg_counter = int(round(sum_counter / num_clients))
            # Clamp to valid range [0, counter_max]
            avg_counter = max(0, min(self.config.counter_max, avg_counter))
            agg_model.predictor.prediction_table[i] = avg_counter
        
        return agg_model

# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================
def evaluate_predictor(predictor: BimodalPredictor, data_loader: DataLoader) -> Tuple[float, float]:
    """
    Evaluate predictor accuracy and MPKI on dataset
    Returns: (accuracy, mpki)
    """
    # Create a fresh predictor for evaluation
    eval_predictor = BimodalPredictor(predictor.config)
    
    correct = 0
    total = 0
    mispredictions = 0
    
    for Xb, yb in data_loader:
        for i in range(Xb.size(0)):
            pc = Xb[i, 0].item()
            actual = yb[i].item()
            
            # Predict
            pred = eval_predictor.predict(pc)
            
            # Update with actual
            eval_predictor.update(pc, actual)
            
            # Count
            if pred == actual:
                correct += 1
            else:
                mispredictions += 1
            total += 1
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    mpki = (mispredictions / total) * 1000 if total > 0 else 0
    
    return accuracy, mpki

# ==========================================================
# LOAD CLIENT DATA
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train_loaders = []
client_test_loaders = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))
    
    client_train_loaders.append(make_loader(X_train, y_train, shuffle=True))
    client_test_loaders.append(make_loader(X_test, y_test, shuffle=False))
    client_test_labels.append(y_test)

# Global test
X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global, shuffle=False)

num_clients = len(client_train_loaders)

# ==========================================================
# INITIALIZE BIMODAL PREDICTOR
# ==========================================================
# Try different Bimodal configurations
configs_to_try = [
    ("Tiny (256 entries, 2-bit)", BimodalConfig(table_size=256, counter_bits=2)),
    ("Small (1KB, 1024 entries)", BimodalConfig(table_size=1024, counter_bits=2)),
    ("Medium (2KB, 2048 entries)", BimodalConfig(table_size=2048, counter_bits=2)),
    ("Large (8KB, 8192 entries)", BimodalConfig(table_size=8192, counter_bits=2)),
    ("X-Large (32KB, 32768 entries)", BimodalConfig(table_size=32768, counter_bits=2)),
]

# Use medium configuration by default (2KB)
config = BimodalConfig(table_size=2048, counter_bits=2)

print(f"\n{'='*60}")
print(f"BIMODAL BRANCH PREDICTOR CONFIGURATION")
print(f"{'='*60}")
print(f"Table size: {config.table_size} entries")
print(f"Counter bits: {config.counter_bits} bits")
print(f"Counter range: 0-{config.counter_max}")
print(f"Prediction threshold: >= {config.midpoint} (taken)")
print(f"Total storage: {config.get_total_storage_kb():.2f} KB")
print(f"Total storage: {config.get_total_storage_bytes()} bytes")
print(f"Index bits: {int(np.log2(config.table_size))} bits")

federated = FederatedBimodal(config)

# ==========================================================
# FEDERATED LEARNING LOOP
# ==========================================================
print(f"\n{'='*60}")
print(f"Starting Federated Learning with Bimodal Predictor")
print(f"{'='*60}")

global_accuracies = []
global_mpki_list = []

for rnd in range(1, FL_ROUNDS + 1):
    print(f"\n{'='*40}")
    print(f"ROUND {rnd}/{FL_ROUNDS}")
    print(f"{'='*40}")
    
    client_models = []
    client_accuracies = []
    client_mpki_list = []
    
    # -------------------
    # CLIENT TRAINING
    # -------------------
    for c in range(num_clients):
        # Create client model
        client_model = BimodalModel(config)
        
        # Train on client data stream
        trained_model = federated.train_client(client_model, client_train_loaders[c])
        
        # Evaluate on client test set
        accuracy, mpki = evaluate_predictor(trained_model.predictor, client_test_loaders[c])
        client_accuracies.append(accuracy)
        client_mpki_list.append(mpki)
        
        client_models.append(trained_model)
        
        print(f"Client {c:2d} | Test Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")
    
    # -------------------
    # AVERAGE CLIENT STATS
    # -------------------
    avg_acc = np.mean(client_accuracies)
    avg_mpki = np.mean(client_mpki_list)
    print(f"\nAvg Client | Acc: {avg_acc:.2f}% | MPKI: {avg_mpki:.2f}")
    
    # -------------------
    # FEDERATED AGGREGATION
    # -------------------
    global_model = federated.aggregate_models(client_models)
    
    # -------------------
    # GLOBAL TEST EVALUATION
    # -------------------
    global_accuracy, global_mpki = evaluate_predictor(global_model.predictor, global_loader)
    global_accuracies.append(global_accuracy)
    global_mpki_list.append(global_mpki)
    
    print(f"\n>>> Global Model | Acc: {global_accuracy:.2f}% | MPKI: {global_mpki:.2f}")
    
    # Update global model for next round
    federated.global_model = global_model

# ==========================================================
# FINAL EVALUATION
# ==========================================================
print(f"\n{'='*60}")
print(f"FINAL EVALUATION")
print(f"{'='*60}")

# Test on all client test sets with final global model
print("\nPer-client evaluation with final global model:")
client_final_accs = []
client_final_mpki = []

for c in range(num_clients):
    accuracy, mpki = evaluate_predictor(global_model.predictor, client_test_loaders[c])
    client_final_accs.append(accuracy)
    client_final_mpki.append(mpki)
    print(f"Client {c:2d} | Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")

print(f"\n{'='*40}")
print(f"CLIENT AVERAGES")
print(f"{'='*40}")
print(f"Average Client Accuracy: {np.mean(client_final_accs):.2f}% ± {np.std(client_final_accs):.2f}")
print(f"Average Client MPKI: {np.mean(client_final_mpki):.2f} ± {np.std(client_final_mpki):.2f}")

# Global test set evaluation
final_accuracy, final_mpki = evaluate_predictor(global_model.predictor, global_loader)
print(f"\n{'='*40}")
print(f"GLOBAL RESULTS")
print(f"{'='*40}")
print(f"Final Global Accuracy: {final_accuracy:.2f}%")
print(f"Final Global MPKI: {final_mpki:.2f}")

# Detailed statistics
total_samples = len(y_global)
correct = int(final_accuracy * total_samples / 100)
print(f"\nDetailed Statistics:")
print(f"  Total samples: {total_samples}")
print(f"  Correct predictions: {correct}")
print(f"  Mispredictions: {total_samples - correct}")

# ==========================================================
# PERFORMANCE METRICS
# ==========================================================
print(f"\n{'='*60}")
print(f"PERFORMANCE SUMMARY")
print(f"{'='*60}")

print(f"\nBimodal Predictor Metrics:")
print(f"  Storage: {config.get_total_storage_kb():.2f} KB ({config.get_total_storage_bytes()} bytes)")
print(f"  Table entries: {config.table_size}")
print(f"  Counter bits: {config.counter_bits} (2-bit saturating)")
print(f"  Indexing: PC modulo table size")
print(f"  Prediction latency: 1 cycle (table lookup + comparison)")

print(f"\nAccuracy Metrics:")
print(f"  Final Global Accuracy: {final_accuracy:.2f}%")
print(f"  Final Global MPKI: {final_mpki:.2f}")
print(f"  Average Client Accuracy: {np.mean(client_final_accs):.2f}%")
print(f"  Average Client MPKI: {np.mean(client_final_mpki):.2f}")

# Compare different configurations
print(f"\n{'='*60}")
print(f"CONFIGURATION COMPARISON")
print(f"{'='*60}")

for name, cfg in configs_to_try:
    print(f"\n{name}:")
    print(f"  Table size: {cfg.table_size} entries")
    print(f"  Storage: {cfg.get_total_storage_kb():.2f} KB")
    print(f"  Index bits: {int(np.log2(cfg.table_size))} bits")

# ==========================================================
# PREDICTION TABLE ANALYSIS
# ==========================================================
def analyze_prediction_table(predictor: BimodalPredictor):
    """Analyze the prediction table distribution"""
    table = predictor.prediction_table
    config = predictor.config
    
    # Count distribution
    counts = predictor.get_table_distribution()
    
    print(f"\n{'='*60}")
    print(f"PREDICTION TABLE ANALYSIS")
    print(f"{'='*60}")
    
    # Map counter values to meanings
    if config.counter_bits == 2:
        meanings = {
            0: "Strongly Not Taken",
            1: "Weakly Not Taken",
            2: "Weakly Taken",
            3: "Strongly Taken"
        }
    else:
        meanings = {i: f"Counter {i}" for i in range(config.counter_max + 1)}
    
    for counter_value, count in counts.items():
        percentage = (count / len(table)) * 100
        meaning = meanings.get(counter_value, f"Counter {counter_value}")
        print(f"  {meaning:20s}: {count:6d} entries ({percentage:5.2f}%)")
    
    # Calculate bias (how many entries predict taken)
    taken_predicting = sum(1 for c in table if c >= config.midpoint)
    not_taken_predicting = len(table) - taken_predicting
    print(f"\n  Bias:")
    print(f"    Predicting Taken: {taken_predicting:6d} entries ({taken_predicting/len(table)*100:.2f}%)")
    print(f"    Predicting Not Taken: {not_taken_predicting:6d} entries ({not_taken_predicting/len(table)*100:.2f}%)")

# Analyze final predictor state
analyze_prediction_table(global_model.predictor)

# ==========================================================
# SAVE MODEL (OPTIONAL)
# ==========================================================
model_path = "bimodal_final.pth"
torch.save({
    'config': {
        'table_size': config.table_size,
        'counter_bits': config.counter_bits
    },
    'prediction_table': global_model.predictor.prediction_table,
    'final_accuracy': final_accuracy,
    'final_mpki': final_mpki
}, model_path)
print(f"\nModel saved to: {model_path}")

# ==========================================================
# CONVERGENCE ANALYSIS
# ==========================================================
print(f"\n{'='*60}")
print(f"CONVERGENCE ANALYSIS")
print(f"{'='*60}")
print(f"Global accuracy over rounds:")
for i, acc in enumerate(global_accuracies[::10]):  # Show every 10th round
    if (i+1)*10 <= len(global_accuracies):
        print(f"  Round {(i+1)*10:2d}: {acc:.2f}%")
print(f"  Final: {global_accuracies[-1]:.2f}%")

# Show MPKI improvement
print(f"\nGlobal MPKI over rounds:")
for i, mpki in enumerate(global_mpki_list[::10]):
    if (i+1)*10 <= len(global_mpki_list):
        print(f"  Round {(i+1)*10:2d}: {mpki:.2f}")
print(f"  Final: {global_mpki_list[-1]:.2f}")

# ==========================================================
# THEORETICAL LIMITS
# ==========================================================
print(f"\n{'='*60}")
print(f"THEORETICAL LIMITS")
print(f"{'='*60}")

# Calculate theoretical maximum accuracy based on branch bias
def calculate_branch_bias(predictor: BimodalPredictor, data_loader: DataLoader) -> float:
    """Calculate the bias of branches in the dataset"""
    taken_count = 0
    total = 0
    
    for Xb, yb in data_loader:
        taken_count += yb.sum().item()
        total += yb.size(0)
    
    bias = taken_count / total if total > 0 else 0.5
    max_accuracy = max(bias, 1 - bias) * 100
    
    return max_accuracy

# Calculate for global test set
max_accuracy = calculate_branch_bias(global_model.predictor, global_loader)
print(f"\nTheoretical maximum accuracy (always predict most common outcome): {max_accuracy:.2f}%")
print(f"Bimodal achieved accuracy: {final_accuracy:.2f}%")
print(f"Gap to theoretical limit: {max_accuracy - final_accuracy:.2f}%")

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")


BIMODAL BRANCH PREDICTOR CONFIGURATION
Table size: 2048 entries
Counter bits: 2 bits
Counter range: 0-3
Prediction threshold: >= 2 (taken)
Total storage: 0.50 KB
Total storage: 512 bytes
Index bits: 11 bits

Starting Federated Learning with Bimodal Predictor

ROUND 1/50
Client  0 | Test Acc: 76.85% | MPKI: 231.54
Client  1 | Test Acc: 81.66% | MPKI: 183.45
Client  2 | Test Acc: 69.89% | MPKI: 301.10
Client  3 | Test Acc: 51.08% | MPKI: 489.20
Client  4 | Test Acc: 62.30% | MPKI: 376.95
Client  5 | Test Acc: 56.60% | MPKI: 433.99

Avg Client | Acc: 66.40% | MPKI: 336.04

>>> Global Model | Acc: 65.78% | MPKI: 342.23

ROUND 2/50
Client  0 | Test Acc: 76.85% | MPKI: 231.54
Client  1 | Test Acc: 81.66% | MPKI: 183.45
Client  2 | Test Acc: 69.89% | MPKI: 301.10
Client  3 | Test Acc: 51.08% | MPKI: 489.20
Client  4 | Test Acc: 62.30% | MPKI: 376.95
Client  5 | Test Acc: 56.60% | MPKI: 433.99

Avg Client | Acc: 66.40% | MPKI: 336.04

>>> Global Model | Acc: 65.78% | MPKI: 342.23

ROUND 3/50


KeyboardInterrupt: 